# Lab 4

This lab covers the lectures from the fourth week of class - 9, 10, 11:
- Transformations/PCA
- Support Vector Machines (SVM)
- Statistical Evaluation

### Learning Objectives
- Understand the basic linear algebra behing PCA
- Apply PCA to the PBMC dataset
- Visually and mathematically explore the kernel trick using polynomial and RBF kernels
- Use cross-validation and statistical evaluations to compare SVM kernels on a real-world dataset
- Evaluate and quantitatively compare models on a classification task of your choice

## Section 1: PCA Transformations
*You should be able to work through this section anytime after Lecture 9*.

Last week, we visualized the PBMC dataset using several t-SNE embeddings, and tried to classify it using a k-NN model. As a reminder, there was a particularly large cluster of T-cells that didn't separate well in the t-SNE plot, and those clusters were difficult to separate in the k-NN model.

The t-SNE plot also took a very long time to generate because the dataset is very large, with 2638 cells and 200 genes (downsampled from the original 13,714).

We're going to try reducing the complexity of this dataset with principal components analysis (PCA) before doing re-applying some of the same techniques as last week.

### 1.2: Centering and Scaling the Data & Initial PCA Visualizations

❓1.2.1

> Since this is gene expression counts data, all the entries are non-negative. Different genes may be expressed to different average levels; these average differences are not necessarily cell-type specific.

❓1.2.2

> More highly expressed genes tend to have higher variance in counts data. For the purposes of this lab, let's assume that we don't want our analysis to be dominated by highly expressed genes, and we think lowly expressed genes may have some important variation.

❓1.2.3

PCA is not inherently a dimensionality reduction technique. However, by selecting less principal components than there are dimensions, we can transform it into a dimensionality reduction tool.

❓1.2.5 In your plot from question 1.2.4, you should see that the first principal component separates the cells in a sort of smear, with CD14 Mononuclear cells on one side and B cells, Naive CD4 and CD8 T cells and FCGR3A Mononuclear cells on the other side. Part of the reason why is already on your plot! The first and second principal components together capture less than 4% of the total variation in the dataset.

One way to get a clearer picture of how the principal components start to separate the data is to make a small pairplot of the first few principal components.

### 1.3: Explained Variance

**On the plot of cumulative explained variance**: This plot seems to tell a different story! While the first 10-20 principal components explain a lot more variance than later components and that "elbow" is still visible on this plot as a slope change, you can see that adding additional dimensions appears to add a lot more information. In fact, we need around 800 dimensions to recover 80% of the variance in the data, and even at 1000 dimensions we haven't quite levelled off into clearly diminishing returns. For context, this dataset started with 2000 genes (dimensions). It seems that there are many principal components explaining very small amounts of variation in the data. *Foreshadowing: Sometimes not all the variation in your dataset is meaningful*.

## Section 2: Calculating PCA

This week in lecture, you learned about Principal Component Analysis, which can be used as a dimensionality reduction tool by finding principal components within the data and taking the ones with the highest explained variance.

Instead of just using scikit-learn, we are going to see how this works calculating it by hand with a simple example.

❓ 2.1

Format $X$ as follows: $\begin{bmatrix}
    x_{1}       & y_{1}\\
    x_{2}       & y_{2}\\
\end{bmatrix}$

❓ 2.2

Find the mean of the first column and subtract that value from each value in the first column. Repeat this for the second column.

❓ 2.3

The covariance between two variables is defined theoretically as:

cov(x,y) = E[(x - μx)(y - μy)]

where:
- E is the expected value
- μx is the mean of x
- μy is the mean of y

### Sample Estimation

For finite samples, we estimate this using:

cov(x,y) = (1/(n-1)) * Σ((xi - μx)(yi - μy))

### Matrix Formulation

When working with centered data matrix X:
- Each row i contains (xi - μx) for all variables
- X contains all our (x - μx) terms pre-computed

### Understanding X^T × X

When we multiply X^T × X:
1. X is an n × p centered matrix (n observations, p variables)
2. X^T is the p × n transpose
3. The multiplication produces a p × p matrix where:
  - Each element (i,j) is the dot product of column i from X^T with column j from X
  - This dot product sums Σ((xi - μx)(xj - μj))
  - Exactly matching our covariance formula!

### The (n-1) Denominator

We divide by (n-1) rather than n because:
- This is Bessel's correction
- It provides an unbiased estimator of the population covariance
- Accounts for one degree of freedom lost in estimating the mean

This matrix formulation efficiently computes the entire covariance matrix in one operation, rather than calculating each covariance separately.

❓ 2.4.1

This is a good link to refer to if you need a refresher on calculating eigenvalues:
https://www.cuemath.com/algebra/eigenvalues/

❓ 2.4.2

Eigenvalues are related to variance. How?

*Hint: Principal components are also related to variance.*

❓ 2.5

Same website as 2.4.1!
https://www.cuemath.com/algebra/eigenvectors/

❓ 2.6

$Answer = X_{centered} * V_{eigenvectors}$

Each eigenvectors should be a column of $V$


❓ 2.7

*Hint: Every PCA for dimension n must have n PCs. How much variance is explained by each PC? Therefore, how many PC can we use for this without loss of information?*

## Section 3: Exploring Kernels

*You should be able to work through this on your own any time after lecture 10.*

This week in lecture, you learned about Support Vector Machines, which use hyperplanes in high dimensional space to separate classes of the training data by maximizing the "margin" or the distance between the hyperplane and the boundary points, or support vectors.

In a linear classifier, distances between data points and the separating hyperplane are expressed using dot products of vectors, which leads the maximization problem of SVM to be expressed in terms of an inner product. Importantly, the data points are always seen in an inner product in the maximization problem and never "alone" as a vector. While we aren't going to get into the details of solving this optimization problem, this means we don't actually need the data points themselves to create a support vector machine! We just need the inner products of data points to express "distances" between them.

**The kernel trick replaces the dot product with some function of our choosing**. This lets us "project" or map data up to higher-dimensional space, without actually explicitly working in the higher-dimensional space.

### 3.1: Polynomial Kernel Math

We have data points $\vec x = \langle x_1,x_2 \rangle$ and $\vec y = \langle y_1,y_2 \rangle$ that we'll represent as vectors in our 2-D input space. Without changing the kernel or input feature space, we can compute the usual dot product, $K(\vec x, \vec y) = \vec x \cdot \vec y = x_1y_1 + x_2y_2$. We can also explicitly project our input data up into a higher dimensional feature space ($\vec x → \phi(\vec x)$). After projecting both our data points up to the higher dimension, their similarity has changed.

❓3.1.1 Compute the projections $\vec x → \phi(\vec x)$ and $\vec y → \phi(\vec y)$ explicitly for $\phi(\vec x ) = \langle x_1^2, x_2^2, \sqrt{2} x_1, \sqrt{2} x_2, \sqrt{2} x_1x_2,1 \rangle$. Then compute the new inner product $K(\vec x, \vec y) = \phi(\vec x) \cdot \phi(\vec y)$ explicitly. *Your answer should be a polynomial in terms of $x_1, x_2, y_1,$ and $y_2$.*

❓3.1.3 In two dimensions with a polynomial kernel of degree 2, computing the projection into higher dimensional space explicitly before taking the dot product was not too much work, and you can imagine the projection as adding columns to a feature matrix in your code with very simple calculations. However, you can see that our 2-D input space ($\vec x \in \mathbb{R}^2$) mapped to a 6-D feature space ($\phi(\vec x) \in \mathbb{R}^6$). [It can be shown](https://en.wikipedia.org/wiki/Polynomial_kernel#Definition) that a polynomial kernel maps to ${n+d}\choose{d}$ dimensions.


**Final Thoughts on the Kernel Trick (post-exercises)**

Just like with polynomial regression, increasing the dimensionality of our feature space with more complex combinations of input variables will increase the flexibility and complexity of nonlinear models we can build with a linear tool like SVM. If we increase to the third degree polynomial, this polynomial kernel becomes $K(\vec x, \vec y) = (\vec x \cdot \vec y + 1)^3$, mapping to a feature space on the order of $n^3$ dimensions.

With real data, the second calculation becomes a *lot* simpler than the first: the calculation is simply a dot product in the input feature space, adding 1, and then cubing the scalar result. If we're only interested in the result of $K$, there is no need to encode our higher dimensional feature space ($\phi (\vec x)$) explicitly.

The power of the kernel trick is twofold:

1) we can create nonlinear decision boundaries. \\
2) we don't have to explicitly compute what our new high-dimensional feature space is.

### 3.2: Visualizing Polynomial Kernels

[This video](https://www.youtube.com/watch?v=OdlNM96sHio) also shows a visualization of the polynomial kernel.

## Section 4: Support Vector Machines

In our first lab, we used multiple logistic regression to classify breast cancer samples as "benign" or "malignant." That dataset happens to be quite linearly separable, as you saw from the classifier doing so well! However, not everything is linearly separable in the dimensions of data you have. If we can map our data to a higher-dimensional space, we can often make our data linearly separable in some higher-dimensional space.

The k-NN classifier we used on the PBMC data last week is a non-linear classifier. While it did pretty well, we're going to try using SVM to improve our classification.

❓ 4.2 Keep this extremely brief. Which kernel performs the best? Which kernel performs the worst? Report this for both accuracy and recall

❓ 4.3 This link explains everything really well for using precision, recall, and accuracy:

https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall